# DementiaSense: Stages 7–12
## Feature engineering, baseline, modelling, tuning, evaluation, use case and limitations

This notebook continues from Stages 2–6. It does **not** re-split the data. It reads the hand-off files that Stages 2–6 already wrote into `outputs/`.

### Do I need to run Stages 2–6 first?

**Only if the hand-off files are missing.** They are already present in this repository (`outputs/train.csv`, `outputs/test.csv`, `outputs/feature_config.json`). Start here. Re-run Stages 2–6 only if those three files are deleted or you change the split, seed, or feature sets.

| File | Role |
|---|---|
| `outputs/train.csv` | Stratified 80% training rows (split in Stage 4, before any fitting) |
| `outputs/test.csv` | Held-out 20% — loaded for convenience, but **not used to make any choice until Stage 11** |
| `outputs/feature_config.json` | Seed (42), target (`Diagnosis`), column types, feature sets A / B / C |

**Working directory:** the DementiaSense repository root (the folder that contains both notebooks and `outputs/`).

### Run order in this notebook

Run **all cells top to bottom**. Do not skip ahead to Stage 11.

| Stage | What you are doing | Test set used to choose anything? |
|---|---|---|
| 7 | Engineered-feature experiments, mutual-information ranking, keep-or-drop decision | No |
| 8 | Four baselines, including the main baseline `LogReg_default` | No |
| 9 | Untuned LogReg, RBF-SVM, Random Forest, HistGradientBoosting vs the baseline | No |
| 10 | `RandomizedSearchCV` on the training folds | No |
| 11 | Pick the final model from CV, choose the threshold on training out-of-fold scores, then score the test set **once** | Predictions only; no selection |
| 12 | Missing-data check, subgroups, inference demo, use case and limitations | Frozen Stage 11 model |

**Scientific rules**
- Features, models, hyperparameters and the decision threshold are chosen with **training-set cross-validation only**.
- Class imbalance is handled with `class_weight="balanced"` inside the pipeline (Stage 5 finding). Rows are never resampled on the test set.
- Outputs are **screening estimates**, not medical diagnoses.

**Run time:** Stage 10 is the slow cell (about 1–2 minutes with `FAST_MODE = False`). Leave `FAST_MODE = False` for report numbers.


In [9]:
import warnings, random, platform, json, functools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy import stats
from scipy.stats import loguniform

import sklearn
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.metrics import (average_precision_score, roc_auc_score, roc_curve, precision_recall_curve,
                             confusion_matrix, f1_score, balanced_accuracy_score, accuracy_score,
                             brier_score_loss)

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", category=UserWarning)

try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

The history saving thread hit an unexpected error (OperationalError('unable to open database file')).History will not be written to the database.


In [2]:
# =========================== CONFIG ===========================
# cwd must be the DementiaSense folder. Stages 2-6 writes the hand-off into outputs/.
def resolve_out_dir():
    seen, p = [], Path.cwd().resolve()
    for _ in range(6):
        seen.append(p / "outputs")
        p = p.parent
    seen.append(Path("outputs"))
    for cand in seen:
        if (cand / "train.csv").exists() and (cand / "feature_config.json").exists():
            return cand
    return Path("outputs")

OUT_DIR = resolve_out_dir()
FIG_DIR = OUT_DIR / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = OUT_DIR / "models"; MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Stage 5 (train-fold CV): PR-AUC barely moves with resampling (~0.39 / 0.60 / 0.85 on A/B/C),
# but class_weight raises LogReg recall (e.g. 0.05 -> 0.53 on set A) without fabricating rows.
# SMOTE is a poor fit: many features are binary/coded, so interpolations are not real patients.
BALANCING = "class_weight"   # "none" | "class_weight" | "smote" | "undersample"
RECALL_TARGET = 0.90         # screening goal; threshold chosen on training OOF, never on test
PRIMARY_SET = "A_screening"  # cheap-to-collect inputs for the Stage 12 screening demo
TUNE_SETS = ["A_screening", "B_plus_symptoms", "C_plus_cognitive_tests"]

FAST_MODE = False            # True = tiny search / few bootstraps, smoke-test only
N_ITER = 3 if FAST_MODE else 15
N_BOOT = 100 if FAST_MODE else 1000
# ==========================================================================================================

if BALANCING in ("smote", "undersample") and not HAS_IMBLEARN:
    raise ImportError("BALANCING needs imbalanced-learn: pip install imbalanced-learn")

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

In [3]:
# ---------- Load the Stage 4 hand-off (created by the Stages 2-6 notebook) ----------
needed = [OUT_DIR / "train.csv", OUT_DIR / "test.csv", OUT_DIR / "feature_config.json"]
missing = [str(p.resolve()) for p in needed if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Stages 7-12 cannot run independently. Execute DementiaSense_Stages_2_to_6.ipynb first "
        "(working directory = this repository). Missing:\n  " + "\n  ".join(missing)
    )

with open(OUT_DIR / "feature_config.json") as f:
    cfg = json.load(f)
SEED = cfg["seed"]; TARGET = cfg["target"]
CONTINUOUS, BINARY, ORDINAL, NOMINAL = cfg["continuous"], cfg["binary"], cfg["ordinal"], cfg["nominal"]
FEATURE_SETS = cfg["feature_sets"]
ALL_FEATURES = CONTINUOUS + BINARY + ORDINAL + NOMINAL
expected_cols = set(ALL_FEATURES + [TARGET])

random.seed(SEED); np.random.seed(SEED)

train = pd.read_csv(OUT_DIR / "train.csv"); test = pd.read_csv(OUT_DIR / "test.csv")
for name, d in (("train.csv", train), ("test.csv", test)):
    missing_cols = expected_cols - set(d.columns)
    extra = set(d.columns) - expected_cols
    if missing_cols:
        raise ValueError(f"{name} is missing columns expected by feature_config.json: {sorted(missing_cols)}")
    if extra:
        print(f"Note: {name} has unused extra columns (ignored): {sorted(extra)}")

X_train, y_train = train[ALL_FEATURES].copy(), train[TARGET].astype(int)
X_test, y_test = test[ALL_FEATURES].copy(), test[TARGET].astype(int)

print("Python", platform.python_version(), "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| seed", SEED)
print("Hand-off directory:", OUT_DIR.resolve())
print(f"Train {X_train.shape} (positive rate {y_train.mean():.3f}) | Test {X_test.shape} (positive rate {y_test.mean():.3f})")
print("Balancing strategy carried over from Stage 5:", BALANCING)
print("Leakage guard: X_test / y_test are loaded but not used for any choice until Stage 11.")

bal_path = OUT_DIR / "stage5_balancing_cv_results.csv"
if bal_path.exists():
    print("\nStage 5 balancing CV (training folds only) — reason for BALANCING = class_weight:")
    display(pd.read_csv(bal_path))

SYMPTOMS = [c for c in FEATURE_SETS["B_plus_symptoms"] if c not in FEATURE_SETS["A_screening"]]
COG_TESTS = [c for c in FEATURE_SETS["C_plus_cognitive_tests"] if c not in FEATURE_SETS["B_plus_symptoms"]]
MODES = {"A_screening": "Screening (demographics, lifestyle, history, clinical measurements)",
         "B_plus_symptoms": "Screening + reported symptoms",
         "C_plus_cognitive_tests": "Decision support (adds MMSE, functional assessment, ADL)"}
print("Symptom features:", SYMPTOMS)
print("Cognitive/functional test features:", COG_TESTS)

Python 3.10.7 | pandas 2.3.3 | scikit-learn 1.7.2 | seed 42
Hand-off directory: /Users/hirenrathva/DementiaSense/outputs
Train (1719, 32) (positive rate 0.354) | Test (430, 32) (positive rate 0.353)
Balancing strategy carried over from Stage 5: class_weight
Leakage guard: X_test / y_test are loaded but not used for any choice until Stage 11.

Stage 5 balancing CV (training folds only) — reason for BALANCING = class_weight:


,feature_set,model,strategy,pr_auc,roc_auc,recall,precision,f1,balanced_acc
0,A_screening,LogReg,none,0.393 +/- 0.018,0.541 +/- 0.011,0.046 +/- 0.024,0.411 +/- 0.082,0.082 +/- 0.040,0.507 +/- 0.008
1,A_screening,LogReg,class_weight,0.394 +/- 0.018,0.541 +/- 0.011,0.534 +/- 0.018,0.398 +/- 0.018,0.456 +/- 0.018,0.546 +/- 0.017
2,A_screening,RandomForest,none,0.382 +/- 0.017,0.523 +/- 0.024,0.048 +/- 0.013,0.415 +/- 0.033,0.085 +/- 0.022,0.506 +/- 0.005
3,A_screening,RandomForest,class_weight,0.400 +/- 0.013,0.518 +/- 0.013,0.051 +/- 0.014,0.642 +/- 0.098,0.094 +/- 0.025,0.518 +/- 0.008
4,B_plus_symptoms,LogReg,none,0.600 +/- 0.045,0.709 +/- 0.032,0.472 +/- 0.036,0.629 +/- 0.059,0.537 +/- 0.029,0.658 +/- 0.021
5,B_plus_symptoms,LogReg,class_weight,0.600 +/- 0.045,0.708 +/- 0.031,0.579 +/- 0.039,0.594 +/- 0.040,0.585 +/- 0.031,0.680 +/- 0.025
6,B_plus_symptoms,RandomForest,none,0.556 +/- 0.028,0.687 +/- 0.019,0.363 +/- 0.031,0.621 +/- 0.046,0.456 +/- 0.013,0.619 +/- 0.002
7,B_plus_symptoms,RandomForest,class_weight,0.565 +/- 0.030,0.685 +/- 0.021,0.344 +/- 0.031,0.635 +/- 0.053,0.444 +/- 0.025,0.617 +/- 0.013
8,C_plus_cognitive_tests,LogReg,none,0.847 +/- 0.027,0.903 +/- 0.013,0.733 +/- 0.044,0.801 +/- 0.045,0.763 +/- 0.017,0.815 +/- 0.013
9,C_plus_cognitive_tests,LogReg,class_weight,0.846 +/- 0.030,0.904 +/- 0.014,0.819 +/- 0.030,0.717 +/- 0.039,0.763 +/- 0.016,0.819 +/- 0.012


Symptom features: ['MemoryComplaints', 'BehavioralProblems', 'Confusion', 'Disorientation', 'PersonalityChanges', 'DifficultyCompletingTasks', 'Forgetfulness']
Cognitive/functional test features: ['MMSE', 'FunctionalAssessment', 'ADL']


---
# Stage 7: Feature engineering and selection

**What this stage does:** propose a few clinically motivated derived features, check whether they are redundant, rank all features by mutual information, then decide with **training-set CV** whether to keep the engineered columns or drop weak features.

**Cells to run:** engineered-feature definitions → redundancy / VIF → mutual-information ranking → CV experiment (`base` vs `base+engineered` vs `base+eng+top15`) → `USE_ENGINEERED` / `SELECT_K` decision.

**Why this stage exists:** the raw table already has many columns. New features are only worth keeping if they improve **PR-AUC** by more than the usual fold-to-fold noise. The test set is not used here.

### Engineered features (row-wise; no learned parameters)

These are calculated independently on train and on test, so they cannot leak labels.

| Feature | Definition | Motivation |
|---|---|---|
| `PulsePressure` | Systolic − Diastolic BP | Arterial stiffness; vascular contribution to cognitive decline |
| `Chol_HDL_Ratio` | Total / HDL cholesterol | Standard cardiovascular risk ratio |
| `LDL_HDL_Ratio` | LDL / HDL cholesterol | Atherogenic balance |
| `NonHDL_Chol` | Total − HDL cholesterol | Remaining atherogenic cholesterol |
| `CardiometabolicCount` | CVD + diabetes + hypertension | Cumulative vascular burden |
| `SymptomCount` | Count of reported symptoms (sets B and C only) | Combines several weak flags into one count |

`SymptomCount` is **not** added to set A, because set A is the cheap screening view that does not include symptom flags.


In [4]:
CARDIO_COLS = ["CardiovascularDisease", "Diabetes", "Hypertension"]
ENG_CLINICAL = ["PulsePressure", "Chol_HDL_Ratio", "LDL_HDL_Ratio", "NonHDL_Chol", "CardiometabolicCount"]
ENG_SYMPTOM = ["SymptomCount"]
ENG_COLS = ENG_CLINICAL + ENG_SYMPTOM
ENG_ADD = {"A_screening": ENG_CLINICAL,
           "B_plus_symptoms": ENG_CLINICAL + ENG_SYMPTOM,
           "C_plus_cognitive_tests": ENG_CLINICAL + ENG_SYMPTOM}

def add_features(d):
    d = d.copy()
    d["PulsePressure"] = d["SystolicBP"] - d["DiastolicBP"]
    hdl = d["CholesterolHDL"].replace(0, np.nan)
    d["Chol_HDL_Ratio"] = d["CholesterolTotal"] / hdl
    d["LDL_HDL_Ratio"] = d["CholesterolLDL"] / hdl
    d["NonHDL_Chol"] = d["CholesterolTotal"] - d["CholesterolHDL"]
    d["CardiometabolicCount"] = d[CARDIO_COLS].sum(axis=1, min_count=1)
    d["SymptomCount"] = d[SYMPTOMS].sum(axis=1, min_count=1)
    return d.replace([np.inf, -np.inf], np.nan)

X_train_F = add_features(X_train)   # row-wise, no fitting: safe to apply to train and test separately
X_test_F = add_features(X_test)

print("Columns before:", X_train.shape[1], "| after adding engineered features:", X_train_F.shape[1])
print("Engineered columns with missing values (train):", X_train_F[ENG_COLS].isna().sum()[lambda s: s > 0].to_dict() or "none")
display(X_train_F[ENG_COLS].describe().T.round(2))

# Does each new feature carry a signal? (train only, same effect-size idea as Stage 6)
rows = []
for c in ENG_COLS:
    a, b = X_train_F.loc[y_train == 1, c].dropna(), X_train_F.loc[y_train == 0, c].dropna()
    U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    rows.append({"feature": c, "median_diagnosed": a.median(), "median_not": b.median(),
                 "rank_biserial": 2 * U / (len(a) * len(b)) - 1, "p_value": p})
display(pd.DataFrame(rows).round(4))

Columns before: 32 | after adding engineered features: 38
Engineered columns with missing values (train): none


,count,mean,std,min,25%,50%,75%,max
PulsePressure,1719.0,43.85,31.20,-28.00,21.00,44.00,66.00,118.00
Chol_HDL_Ratio,1719.0,4.62,2.43,1.54,2.84,3.88,5.78,14.27
LDL_HDL_Ratio,1719.0,2.57,1.61,0.52,1.43,2.11,3.25,9.45
NonHDL_Chol,1719.0,166.43,48.59,53.38,130.16,166.52,202.16,278.33
CardiometabolicCount,1719.0,0.44,0.61,0.00,0.00,0.00,1.00,3.00
SymptomCount,1719.0,1.33,1.02,0.00,1.00,1.00,2.00,5.00


,feature,median_diagnosed,median_not,rank_biserial,p_value
0,PulsePressure,42.0000,46.0000,-0.0509,0.0808
1,Chol_HDL_Ratio,3.6749,3.9814,-0.0600,0.0396
2,LDL_HDL_Ratio,2.0302,2.1554,-0.0580,0.0463
3,NonHDL_Chol,164.4127,167.5569,-0.0196,0.5020
4,CardiometabolicCount,0.0000,0.0000,0.0088,0.7234
5,SymptomCount,1.0000,1.0000,0.2086,0.0000


In [5]:
# ---------- Redundancy check: engineered features vs the raw features they come from ----------
groups = {"BP": ["SystolicBP", "DiastolicBP", "PulsePressure"],
          "Cholesterol": ["CholesterolTotal", "CholesterolLDL", "CholesterolHDL", "CholesterolTriglycerides",
                          "NonHDL_Chol", "Chol_HDL_Ratio", "LDL_HDL_Ratio"]}
for name, cols in groups.items():
    corr = X_train_F[cols].corr(method="spearman")
    Z = X_train_F[cols].fillna(X_train_F[cols].median())
    vif = pd.Series(np.diag(np.linalg.pinv(np.corrcoef(Z.values.T))), index=cols, name="VIF")
    print(f"--- {name} ---")
    display(corr.round(2))
    display(vif.round(1).to_frame().T)
print("NonHDL_Chol and PulsePressure are exact linear combinations of raw columns, so they add redundancy for linear models.")
print("Regularised logistic regression and tree models cope with this; it is still worth testing, which is what 7.3 does.")

--- BP ---


,SystolicBP,DiastolicBP,PulsePressure
SystolicBP,1.00,0.00,0.83
DiastolicBP,0.00,1.00,-0.53
PulsePressure,0.83,-0.53,1.00


,SystolicBP,DiastolicBP,PulsePressure
VIF,0.5,0.8,0.3


--- Cholesterol ---


,CholesterolTotal,CholesterolLDL,CholesterolHDL,CholesterolTriglycerides,NonHDL_Chol,Chol_HDL_Ratio,LDL_HDL_Ratio
CholesterolTotal,1.00,0.00,-0.00,0.00,0.88,0.40,0.01
CholesterolLDL,0.00,1.00,-0.04,-0.02,0.02,0.04,0.66
CholesterolHDL,-0.00,-0.04,1.00,0.01,-0.46,-0.91,-0.76
CholesterolTriglycerides,0.00,-0.02,0.01,1.00,-0.01,-0.01,-0.02
NonHDL_Chol,0.88,0.02,-0.46,-0.01,1.00,0.77,0.35
Chol_HDL_Ratio,0.40,0.04,-0.91,-0.01,0.77,1.00,0.70
LDL_HDL_Ratio,0.01,0.66,-0.76,-0.02,0.35,0.70,1.00


,CholesterolTotal,CholesterolLDL,CholesterolHDL,CholesterolTriglycerides,NonHDL_Chol,Chol_HDL_Ratio,LDL_HDL_Ratio
VIF,0.8,4.6,4.0,1.0,1.2,12.7,11.9


NonHDL_Chol and PulsePressure are exact linear combinations of raw columns, so they add redundancy for linear models.
Regularised logistic regression and tree models cope with this; it is still worth testing, which is what 7.3 does.


In [6]:
# ---------- Feature relevance ranking: mutual information (train only) ----------
CONT_ALL = CONTINUOUS + ENG_COLS

def build_preprocessor(features):
    cont = [c for c in CONT_ALL if c in features]
    binr = [c for c in BINARY if c in features]
    ordn = [c for c in ORDINAL if c in features]
    nomn = [c for c in NOMINAL if c in features]
    parts = []
    if cont:
        parts.append(("cont", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), cont))
    if binr:
        parts.append(("bin", SimpleImputer(strategy="most_frequent"), binr))
    if ordn:
        parts.append(("ord", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("sc", StandardScaler())]), ordn))
    if nomn:
        parts.append(("nom", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                       ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), nomn))
    return ColumnTransformer(parts, remainder="drop", verbose_feature_names_out=False)

full_feats = FEATURE_SETS["C_plus_cognitive_tests"] + ENG_COLS
pp = build_preprocessor(full_feats)
Xt = pp.fit_transform(X_train_F[full_feats])
mi = pd.Series(mutual_info_classif(Xt, y_train, random_state=SEED), index=pp.get_feature_names_out()).sort_values(ascending=False)
display(mi.head(15).round(4).to_frame("mutual_information"))

fig, ax = plt.subplots(figsize=(7, 6))
mi.head(20)[::-1].plot.barh(ax=ax, color="#5b8fb9"); ax.set_xlabel("Mutual information with Diagnosis (train)")
savefig("07_mutual_information")
print("Features with ~zero mutual information are candidates for removal; 7.3 tests whether removal helps.")

,mutual_information
FunctionalAssessment,0.0861
ADL,0.0773
MMSE,0.0679
MemoryComplaints,0.0453
Diabetes,0.0267
Smoking,0.0208
SymptomCount,0.0161
LDL_HDL_Ratio,0.0137
AlcoholConsumption,0.0119
Ethnicity_2,0.0116


Features with ~zero mutual information are candidates for removal; 7.3 tests whether removal helps.


### 7.3 Training-set experiment: engineered features and top-15 selection

Each variant is scored with 5-fold stratified CV on the **training set**. `SelectKBest` (mutual information) sits **inside** the pipeline, so it is re-fitted on each training fold and never sees the held-out fold.

**Headline metric:** PR-AUC (average precision). It does not depend on a threshold and focuses on the diagnosed class. A change is adopted only if the **mean gain is larger than the typical fold SD**.

| Variant | Meaning |
|---|---|
| `base` | Original features of that set |
| `base+engineered` | Original features plus the engineered columns allowed for that set |
| `base+eng+top15` | Engineered columns, then keep the 15 highest mutual-information features |

The experiment uses Logistic Regression and Random Forest as two different model families (linear vs trees). The decision is then frozen for Stages 8–12.


In [7]:
MODEL_NAMES = ["LogReg", "SVM_RBF", "RandomForest", "HistGB"]
SIMPLICITY = ["LogReg", "SVM_RBF", "RandomForest", "HistGB"]   # simplest first (used for tie-breaking in Stage 11)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = {"pr_auc": "average_precision", "roc_auc": "roc_auc", "recall": "recall",
           "precision": "precision", "f1": "f1", "balanced_acc": "balanced_accuracy"}
MI_SCORE = functools.partial(mutual_info_classif, random_state=SEED)
SELECT_K = None       # set after Stage 7.3 (None = use all features)

def make_model(name, class_weight=None):
    if name == "LogReg":
        return LogisticRegression(max_iter=3000, class_weight=class_weight, random_state=SEED)
    if name == "SVM_RBF":
        return SVC(kernel="rbf", probability=True, class_weight=class_weight, random_state=SEED)
    if name == "RandomForest":
        return RandomForestClassifier(n_estimators=300, class_weight=class_weight, random_state=SEED)
    if name == "HistGB":
        return HistGradientBoostingClassifier(class_weight=class_weight, random_state=SEED)
    raise ValueError(name)

def make_pipeline(features, model_name, strategy=None, select_k="global"):
    strategy = BALANCING if strategy is None else strategy
    k = SELECT_K if select_k == "global" else select_k
    steps = [("prep", build_preprocessor(features))]
    if k:
        k = int(k)
        n_in = len(features)
        steps.append(("select", SelectKBest(MI_SCORE, k=min(k, n_in))))
    if strategy == "smote":
        steps.append(("smote", SMOTE(random_state=SEED)))
    elif strategy == "undersample":
        steps.append(("under", RandomUnderSampler(random_state=SEED)))
    steps.append(("clf", make_model(model_name, "balanced" if strategy == "class_weight" else None)))
    return (ImbPipeline if strategy in ("smote", "undersample") else Pipeline)(steps)

def run_cv(pipe, X, y):
    r = cross_validate(pipe, X, y, cv=CV, scoring=SCORING, return_train_score=True,
                       n_jobs=-1, error_score="raise")
    out = {}
    for m in SCORING:
        out[m] = r[f"test_{m}"].mean(); out[m + "_sd"] = r[f"test_{m}"].std()
    out["train_pr_auc"] = r["train_pr_auc"].mean()
    return out

def show_cv(df, id_cols):
    out = df[id_cols].copy()
    for m in SCORING:
        out[m] = df[m].round(3).astype(str) + " +/- " + df[m + "_sd"].round(3).astype(str)
    out["train_pr_auc"] = df["train_pr_auc"].round(3)
    out["train-val gap"] = (df["train_pr_auc"] - df["pr_auc"]).round(3)
    return out

rows = []
for set_name, base in FEATURE_SETS.items():
    variants = {"base": (base, None),
                "base+engineered": (base + ENG_ADD[set_name], None),
                "base+eng+top15": (base + ENG_ADD[set_name], 15)}
    for model_name in ["LogReg", "RandomForest"]:
        for vname, (feats, k) in variants.items():
            pipe = make_pipeline(feats, model_name, select_k=k)
            rows.append({"feature_set": set_name, "model": model_name, "variant": vname, **run_cv(pipe, X_train_F[feats], y_train)})
exp7 = pd.DataFrame(rows)
display(show_cv(exp7, ["feature_set", "model", "variant"]))
exp7.to_csv(OUT_DIR / "stage7_feature_experiment.csv", index=False)

,feature_set,model,variant,pr_auc,roc_auc,recall,precision,f1,balanced_acc,train_pr_auc,train-val gap
0,A_screening,LogReg,base,0.394 +/- 0.018,0.541 +/- 0.011,0.534 +/- 0.018,0.398 +/- 0.018,0.456 +/- 0.018,0.546 +/- 0.017,0.436,0.042
1,A_screening,LogReg,base+engineered,0.403 +/- 0.024,0.545 +/- 0.015,0.546 +/- 0.023,0.387 +/- 0.024,0.453 +/- 0.024,0.536 +/- 0.025,0.447,0.044
2,A_screening,LogReg,base+eng+top15,0.4 +/- 0.023,0.549 +/- 0.008,0.548 +/- 0.022,0.388 +/- 0.012,0.454 +/- 0.014,0.537 +/- 0.012,0.421,0.021
3,A_screening,RandomForest,base,0.394 +/- 0.017,0.516 +/- 0.009,0.049 +/- 0.014,0.564 +/- 0.097,0.091 +/- 0.024,0.514 +/- 0.008,1.000,0.606
4,A_screening,RandomForest,base+engineered,0.389 +/- 0.014,0.521 +/- 0.008,0.054 +/- 0.006,0.526 +/- 0.072,0.098 +/- 0.012,0.514 +/- 0.005,1.000,0.611
5,A_screening,RandomForest,base+eng+top15,0.388 +/- 0.014,0.533 +/- 0.022,0.058 +/- 0.017,0.398 +/- 0.075,0.1 +/- 0.028,0.505 +/- 0.008,1.000,0.612
6,B_plus_symptoms,LogReg,base,0.6 +/- 0.045,0.708 +/- 0.031,0.579 +/- 0.039,0.594 +/- 0.04,0.585 +/- 0.031,0.68 +/- 0.025,0.621,0.022
7,B_plus_symptoms,LogReg,base+engineered,0.604 +/- 0.041,0.712 +/- 0.034,0.584 +/- 0.038,0.587 +/- 0.044,0.585 +/- 0.034,0.679 +/- 0.028,0.627,0.023
8,B_plus_symptoms,LogReg,base+eng+top15,0.605 +/- 0.036,0.711 +/- 0.03,0.569 +/- 0.043,0.605 +/- 0.035,0.586 +/- 0.03,0.682 +/- 0.022,0.609,0.004
9,B_plus_symptoms,RandomForest,base,0.566 +/- 0.032,0.686 +/- 0.018,0.349 +/- 0.036,0.62 +/- 0.055,0.444 +/- 0.025,0.614 +/- 0.012,1.000,0.434


In [8]:
# ---------- Decision aid ----------
piv = exp7.pivot_table(index=["feature_set", "model"], columns="variant", values="pr_auc")
sd = exp7[exp7.variant == "base"].set_index(["feature_set", "model"])["pr_auc_sd"]
gain_eng = (piv["base+engineered"] - piv["base"])
gain_sel = (piv["base+eng+top15"] - piv["base+engineered"])
display(pd.DataFrame({"PR-AUC gain: engineered vs base": gain_eng, "PR-AUC gain: top15 vs all": gain_sel, "fold SD (base)": sd}).round(4))

SUGGEST_ENG = bool(gain_eng.mean() > sd.mean())
SUGGEST_K = 15 if bool(gain_sel.mean() > sd.mean()) else None
print(f"Mean gain from engineered features: {gain_eng.mean():+.4f} (typical fold SD {sd.mean():.4f}) -> suggest keeping them: {SUGGEST_ENG}")
print(f"Mean gain from top-15 selection:    {gain_sel.mean():+.4f} -> suggest using selection: {SUGGEST_K is not None}")
print("Rule used: adopt a change only if the average gain exceeds the typical fold-to-fold spread. Override below if you can justify it.")

PR-AUC gain: engineered vs base  PR-AUC gain: top15 vs all  fold SD (base)
feature_set            model                                                                                   
A_screening            LogReg                                 0.0085                    -0.0025          0.0179
                       RandomForest                          -0.0049                    -0.0012          0.0166
B_plus_symptoms        LogReg                                 0.0040                     0.0013          0.0447
                       RandomForest                          -0.0017                    -0.0097          0.0317
C_plus_cognitive_tests LogReg                                 0.0018                     0.0025          0.0296
                       RandomForest                          -0.0078                    -0.0009          0.0163

Mean gain from engineered features: -0.0000 (typical fold SD 0.0261) -> suggest keeping them: False
Mean gain from top-15 selection:    -0.0017 -> suggest using selection: False
Rule used: adopt a change only if the average gain exceeds the typical fold-to-fold spread. Override below if you can justify it.


In [9]:
# ---------- Stage 7 decision: follow the CV gain rule unless a written clinical reason overrides it ----------
USE_ENGINEERED = bool(SUGGEST_ENG)
SELECT_K = None if SUGGEST_K in (None, False) else int(SUGGEST_K)

FINAL_SETS = {name: list(base) + (ENG_ADD[name] if USE_ENGINEERED else []) for name, base in FEATURE_SETS.items()}
print("USE_ENGINEERED =", USE_ENGINEERED, "| SELECT_K =", SELECT_K)
for k, v in FINAL_SETS.items():
    print(f"  {k}: {len(v)} features entering the pipeline")
with open(OUT_DIR / "stage7_decisions.json", "w") as f:
    json.dump({"use_engineered": USE_ENGINEERED, "select_k": SELECT_K, "balancing": BALANCING,
               "final_sets": FINAL_SETS}, f, indent=2)


USE_ENGINEERED = False | SELECT_K = None
  A_screening: 22 features entering the pipeline
  B_plus_symptoms: 29 features entering the pipeline
  C_plus_cognitive_tests: 32 features entering the pipeline


### Stage 7 result (from the cells above)

**What we created.** Six derived columns: pulse pressure, two cholesterol ratios, non-HDL cholesterol, a cardiometabolic count, and (for sets B/C) a symptom count.

**Univariate signal (train only).** `SymptomCount` is the only engineered feature with a clear difference between classes (rank-biserial ≈ 0.21, p < 0.001). The cholesterol ratios are barely associated (p ≈ 0.04) and `PulsePressure`, `NonHDL_Chol` and `CardiometabolicCount` are not.

**Redundancy.** `PulsePressure` and `NonHDL_Chol` are linear combinations of columns we already have. The cholesterol ratios have high VIF (about 12), so they largely repeat HDL / LDL / total cholesterol.

**Mutual information (train only).** The ranking is dominated by features that are *close to diagnosis*: `FunctionalAssessment`, `ADL`, `MMSE`, then `MemoryComplaints`. Most lifestyle and lab variables sit near zero. That already hints that cheap screening inputs (set A) will be weak.

**CV decision.** Mean PR-AUC gain from adding engineered features: **0.000** (typical fold SD **0.026**). Mean gain from keeping only the top 15: **−0.002**. Neither change beats fold noise, so:

- `USE_ENGINEERED = False`
- `SELECT_K = None` (use every original column in the set)

Sets A / B / C therefore stay at 22 / 29 / 32 original features. This is an honest negative result: the extra columns were clinically motivated, but they did not improve ranking quality on this dataset.


---
# Stage 8: Baseline models

**Exact cell to run:** the Stage 8 code cell that defines `baseline_pipe` and loops over `Dummy_prior`, `Dummy_stratified`, `DecisionTree_depth3` and `LogReg_default`. It writes `outputs/stage8_baseline_cv.csv`.

**Purpose:** a reference that later models must beat **by more than the fold SD**. All four baselines use the **original Stage 2–6 features only** — no engineered columns, no class weighting, no hyperparameter search. The test set is still unused.

| Baseline | What it establishes |
|---|---|
| `Dummy_prior` | Always predicts the majority class (no diagnosis). Shows “accuracy without skill”: recall of the diagnosed class is 0 |
| `Dummy_stratified` | Random guesses that match the training class frequencies |
| `DecisionTree_depth3` | A tiny, readable rule set — is there a simple threshold on a few columns? |
| `LogReg_default` | **Main baseline:** default logistic regression, threshold 0.5 later, no balancing |

**How to read the metrics (used from here onward)**

| Metric | Plain meaning |
|---|---|
| **PR-AUC** | How well the model ranks diagnosed patients above others. A no-skill model scores about the prevalence (~0.35) |
| **ROC-AUC** | Ranking of diagnosed vs not; 0.5 is chance |
| **Recall** | Of people who truly have the label, what fraction we catch |
| **Precision** | Of people we flag, what fraction truly have the label |
| **F1** | Balance of recall and precision |
| **Balanced accuracy** | Average of recall and specificity; 0.5 is chance |

Accuracy is not used as a headline number because predicting “no dementia” for everyone would look accurate (~65%) while missing every case.


In [10]:
def baseline_pipe(features, name):
    prep = build_preprocessor(features)
    if name == "LogReg_default":
        return make_pipeline(features, "LogReg", strategy="none", select_k=None)
    clf = {"Dummy_prior": DummyClassifier(strategy="prior"),
           "Dummy_stratified": DummyClassifier(strategy="stratified", random_state=SEED),
           "DecisionTree_depth3": DecisionTreeClassifier(max_depth=3, random_state=SEED)}[name]
    return Pipeline([("prep", prep), ("clf", clf)])

rows = []
for set_name, feats in FEATURE_SETS.items():
    for b in ["Dummy_prior", "Dummy_stratified", "DecisionTree_depth3", "LogReg_default"]:
        rows.append({"feature_set": set_name, "baseline": b, **run_cv(baseline_pipe(feats, b), X_train_F[feats], y_train)})
exp8 = pd.DataFrame(rows)
display(show_cv(exp8, ["feature_set", "baseline"]))
exp8.to_csv(OUT_DIR / "stage8_baseline_cv.csv", index=False)

print("Positive-class prevalence in train (= PR-AUC of a model with no skill):", round(y_train.mean(), 3))

,feature_set,baseline,pr_auc,roc_auc,recall,precision,f1,balanced_acc,train_pr_auc,train-val gap
0,A_screening,Dummy_prior,0.354 +/- 0.001,0.5 +/- 0.0,0.0 +/- 0.0,0.0 +/- 0.0,0.0 +/- 0.0,0.5 +/- 0.0,0.354,0.000
1,A_screening,Dummy_stratified,0.353 +/- 0.01,0.497 +/- 0.023,0.339 +/- 0.03,0.349 +/- 0.031,0.344 +/- 0.031,0.497 +/- 0.023,0.356,0.003
2,A_screening,DecisionTree_depth3,0.346 +/- 0.012,0.476 +/- 0.023,0.021 +/- 0.024,0.212 +/- 0.194,0.037 +/- 0.04,0.494 +/- 0.007,0.421,0.075
3,A_screening,LogReg_default,0.393 +/- 0.018,0.541 +/- 0.011,0.046 +/- 0.024,0.411 +/- 0.082,0.082 +/- 0.04,0.507 +/- 0.008,0.438,0.045
4,B_plus_symptoms,Dummy_prior,0.354 +/- 0.001,0.5 +/- 0.0,0.0 +/- 0.0,0.0 +/- 0.0,0.0 +/- 0.0,0.5 +/- 0.0,0.354,0.000
5,B_plus_symptoms,Dummy_stratified,0.353 +/- 0.01,0.497 +/- 0.023,0.339 +/- 0.03,0.349 +/- 0.031,0.344 +/- 0.031,0.497 +/- 0.023,0.356,0.003
6,B_plus_symptoms,DecisionTree_depth3,0.515 +/- 0.031,0.681 +/- 0.026,0.429 +/- 0.101,0.616 +/- 0.059,0.497 +/- 0.06,0.638 +/- 0.032,0.568,0.054
7,B_plus_symptoms,LogReg_default,0.6 +/- 0.045,0.709 +/- 0.032,0.472 +/- 0.036,0.629 +/- 0.059,0.537 +/- 0.029,0.658 +/- 0.021,0.624,0.025
8,C_plus_cognitive_tests,Dummy_prior,0.354 +/- 0.001,0.5 +/- 0.0,0.0 +/- 0.0,0.0 +/- 0.0,0.0 +/- 0.0,0.5 +/- 0.0,0.354,0.000
9,C_plus_cognitive_tests,Dummy_stratified,0.353 +/- 0.01,0.497 +/- 0.023,0.339 +/- 0.03,0.349 +/- 0.031,0.344 +/- 0.031,0.497 +/- 0.023,0.356,0.003


Positive-class prevalence in train (= PR-AUC of a model with no skill): 0.354


### Stage 8 result (training CV)

Positive-class prevalence in train is **0.354**. That is also the PR-AUC of a model with no ranking skill, and it is exactly what both dummy baselines achieve (PR-AUC ≈ 0.35, ROC-AUC ≈ 0.50).

| Feature set | `LogReg_default` PR-AUC | Recall | F1 | Reading |
|---|---|---|---|---|
| **A screening** | **0.393 ± 0.018** | 0.046 | 0.082 | Barely above chance. Demographics, lifestyle and labs almost do not rank diagnosis |
| **B + symptoms** | **0.600 ± 0.045** | 0.472 | 0.537 | Memory complaints and behavioural flags add real signal |
| **C + cognitive tests** | **0.847 ± 0.027** | 0.733 | 0.763 | MMSE, functional assessment and ADL dominate |

The depth-3 tree **fails on set A** (PR-AUC 0.346, worse than the dummy). On set C it is strong (PR-AUC 0.870) because a shallow tree can split on MMSE / ADL / functional scores.

**What later models must beat:** the `LogReg_default` PR-AUC on that same feature set, by more than the fold SD. The jump A → B → C is evidence that *which inputs you are allowed to use* matters more than the algorithm. Set C is closer to a clinical assessment than to cheap community screening — that distinction is the project’s main scientific point.


---
# Stage 9: ML model comparison (untuned)

**Exact cell to run:** the Stage 9 code cell that loops `MODEL_NAMES` over `TUNE_SETS`, then the bar-chart cell. It writes `outputs/stage9_model_comparison_cv.csv` and `outputs/figures/09_model_comparison.png`.

This is still **training-set CV**. Models use the Stage 7 feature decision (original columns only) and Stage 5 balancing (`class_weight`). A model “beats the baseline” only if its CV PR-AUC gain is larger than its own fold SD. Ranking uses **validation** PR-AUC, not the training score.

| Model | Idea in one sentence | Fit for this table |
|---|---|---|
| **Logistic regression (L2)** | A weighted sum of features → probability | Interpretable; needs scaling; natural step up from the baseline |
| **SVM (RBF kernel)** | Finds a smooth non-linear boundary | Can model curves; slow; probabilities come from extra internal CV |
| **Random forest** | Many deep trees, majority vote | Handles interactions; can memorise noise on ~1.7k rows |
| **HistGradientBoosting** | Trees added one after another to correct mistakes | Often strongest on tables; more knobs; overfit risk |

**Deep learning is not used.** There are about 1,719 training rows of mixed tabular features. A neural net is unlikely to beat boosted trees here, is harder to explain, and the assignment allows DL only where it is justified.

Training PR-AUC is printed next to CV PR-AUC so overfitting is visible. A high training score is **not** treated as success.


In [11]:
rows = []
for set_name in TUNE_SETS:
    feats = FINAL_SETS[set_name]
    for m in MODEL_NAMES:
        rows.append({"feature_set": set_name, "model": m, **run_cv(make_pipeline(feats, m), X_train_F[feats], y_train)})
exp9 = pd.DataFrame(rows)
display(show_cv(exp9, ["feature_set", "model"]))
exp9.to_csv(OUT_DIR / "stage9_model_comparison_cv.csv", index=False)

# Compare against the Stage 8 baseline
base_ref = exp8[exp8.baseline == "LogReg_default"].set_index("feature_set")["pr_auc"]
cmp9 = exp9.assign(baseline_pr_auc=exp9.feature_set.map(base_ref))
cmp9["gain_vs_baseline"] = cmp9["pr_auc"] - cmp9["baseline_pr_auc"]
display(cmp9[["feature_set", "model", "pr_auc", "pr_auc_sd", "baseline_pr_auc", "gain_vs_baseline"]].round(3))

,feature_set,model,pr_auc,roc_auc,recall,precision,f1,balanced_acc,train_pr_auc,train-val gap
0,A_screening,LogReg,0.394 +/- 0.018,0.541 +/- 0.011,0.534 +/- 0.018,0.398 +/- 0.018,0.456 +/- 0.018,0.546 +/- 0.017,0.436,0.042
1,A_screening,SVM_RBF,0.386 +/- 0.018,0.517 +/- 0.007,0.447 +/- 0.029,0.367 +/- 0.012,0.402 +/- 0.011,0.512 +/- 0.011,0.798,0.412
2,A_screening,RandomForest,0.394 +/- 0.017,0.516 +/- 0.009,0.049 +/- 0.014,0.564 +/- 0.097,0.091 +/- 0.024,0.514 +/- 0.008,1.000,0.606
3,A_screening,HistGB,0.362 +/- 0.007,0.497 +/- 0.019,0.296 +/- 0.034,0.353 +/- 0.024,0.321 +/- 0.028,0.5 +/- 0.016,1.000,0.638
4,B_plus_symptoms,LogReg,0.6 +/- 0.045,0.708 +/- 0.031,0.579 +/- 0.039,0.594 +/- 0.04,0.585 +/- 0.031,0.68 +/- 0.025,0.621,0.022
5,B_plus_symptoms,SVM_RBF,0.569 +/- 0.045,0.683 +/- 0.02,0.53 +/- 0.033,0.545 +/- 0.035,0.537 +/- 0.031,0.643 +/- 0.024,0.856,0.287
6,B_plus_symptoms,RandomForest,0.566 +/- 0.032,0.686 +/- 0.018,0.349 +/- 0.036,0.62 +/- 0.055,0.444 +/- 0.025,0.614 +/- 0.012,1.000,0.434
7,B_plus_symptoms,HistGB,0.546 +/- 0.025,0.664 +/- 0.016,0.497 +/- 0.037,0.548 +/- 0.044,0.519 +/- 0.019,0.634 +/- 0.016,1.000,0.454
8,C_plus_cognitive_tests,LogReg,0.846 +/- 0.03,0.904 +/- 0.014,0.819 +/- 0.03,0.717 +/- 0.039,0.763 +/- 0.016,0.819 +/- 0.012,0.860,0.014
9,C_plus_cognitive_tests,SVM_RBF,0.842 +/- 0.034,0.894 +/- 0.016,0.799 +/- 0.02,0.736 +/- 0.044,0.765 +/- 0.026,0.82 +/- 0.02,0.960,0.118


,feature_set,model,pr_auc,pr_auc_sd,baseline_pr_auc,gain_vs_baseline
0,A_screening,LogReg,0.394,0.018,0.393,0.001
1,A_screening,SVM_RBF,0.386,0.018,0.393,-0.008
2,A_screening,RandomForest,0.394,0.017,0.393,0.001
3,A_screening,HistGB,0.362,0.007,0.393,-0.032
4,B_plus_symptoms,LogReg,0.600,0.045,0.600,-0.000
5,B_plus_symptoms,SVM_RBF,0.569,0.045,0.600,-0.031
6,B_plus_symptoms,RandomForest,0.566,0.032,0.600,-0.034
7,B_plus_symptoms,HistGB,0.546,0.025,0.600,-0.054
8,C_plus_cognitive_tests,LogReg,0.846,0.030,0.847,-0.001
9,C_plus_cognitive_tests,SVM_RBF,0.842,0.034,0.847,-0.006


In [12]:
piv9 = exp9.pivot(index="model", columns="feature_set", values="pr_auc")[TUNE_SETS]
piv9.plot.bar(figsize=(8, 4), rot=0); plt.ylabel("CV PR-AUC (train folds)")
plt.axhline(y_train.mean(), color="grey", ls="--", lw=1, label="no-skill (prevalence)")
plt.legend(fontsize=8); plt.title("Model comparison before tuning")
savefig("09_model_comparison")

### Stage 9 result (untuned, training CV)

Compare each model with Stage 8 `LogReg_default` (PR-AUC 0.393 / 0.600 / 0.847 on A / B / C).

**Set A (screening).** Nobody beats the baseline in a meaningful way. LogReg 0.394 and Random Forest 0.394 match it; SVM 0.386 and HistGB 0.362 are worse. ROC-AUC stays around 0.50–0.54. **The limit is the features, not the algorithm.** Trees show a huge train–validation gap (train PR-AUC = 1.0, CV ≈ 0.36–0.39): they memorise the training rows.

**Set B (symptoms added).** LogReg 0.600 **ties** the baseline. SVM, Random Forest and HistGB are all *below* the baseline (0.569, 0.566, 0.546). Class weighting raises LogReg recall (0.579 vs 0.472 in the unweighted baseline) without changing PR-AUC. Again the linear model is enough.

**Set C (cognitive tests added).** Here the trees **do** beat the baseline by more than the fold SD: Random Forest **0.937** (gain +0.089) and HistGB **0.943** (gain +0.096). LogReg 0.846 and SVM 0.842 match the baseline. So non-linear models help only once MMSE, functional assessment and ADL are available.

**Takeaway for the viva.** On cheap inputs, pick the simplest model. On clinic-style cognitive scores, tree ensembles rank much better — but those scores are collected late in the pathway, so they are not an early-screening win.


---
# Stage 10: Randomized hyperparameter search

**Exact cell to run:** the Stage 10 cell that builds `PARAM_DISTS` and fits `RandomizedSearchCV` (the slow cell), then the summary cell. It writes `outputs/stage10_tuning_summary.csv` and `outputs/stage10_best_params.json`.

**Method.** Randomised search with the same 5-fold stratified CV on the **training set**. Random search covers wide ranges more efficiently than a full grid, especially for continuous parameters (C, gamma, learning rate).

| Model | What is searched | Why those knobs |
|---|---|---|
| LogReg | `C` on a log scale from 0.001 to 100 | Smaller C = stronger regularisation (safer on noisy, weak features) |
| SVM-RBF | `C` and `gamma` | C is the penalty; gamma controls how wiggly the boundary is |
| Random Forest | trees, depth, leaf size, `max_features` | Depth and leaf size control memorisation |
| HistGB | learning rate, leaves, `max_iter`, L2 | Slow learning plus fewer leaves reduces overfit |

**Selection metric:** PR-AUC (`average_precision`). Recall and F1 are recorded but do not pick the winner. The **decision threshold** is chosen later in Stage 11, not here.

**Caution:** the best CV score from a search is slightly optimistic (it is the maximum over many candidates). Stage 11’s held-out test set is the real check. `N_ITER = 15` (`FAST_MODE = False`).


In [13]:
PARAM_DISTS = {
    "LogReg": {"clf__C": loguniform(1e-3, 1e2)},
    "SVM_RBF": {"clf__C": loguniform(1e-2, 1e2), "clf__gamma": loguniform(1e-4, 1e0)},
    "RandomForest": {"clf__n_estimators": [100, 200, 300], "clf__max_depth": [None, 4, 6, 8, 12],
                     "clf__min_samples_leaf": [1, 2, 5, 10, 20], "clf__max_features": ["sqrt", "log2", 0.5]},
    "HistGB": {"clf__learning_rate": loguniform(0.01, 0.3), "clf__max_leaf_nodes": [8, 15, 31],
               "clf__min_samples_leaf": [10, 20, 40], "clf__l2_regularization": loguniform(1e-3, 10),
               "clf__max_iter": [100, 200, 300]},
}

tuned = {}
for set_name in TUNE_SETS:
    feats = FINAL_SETS[set_name]
    for m in MODEL_NAMES:
        s = RandomizedSearchCV(make_pipeline(feats, m), PARAM_DISTS[m], n_iter=N_ITER, scoring=SCORING,
                               refit="pr_auc", cv=CV, n_jobs=-1, random_state=SEED, error_score="raise")
        s.fit(X_train_F[feats], y_train)
        tuned[(set_name, m)] = s
        print(f"done: {set_name:24s} {m:13s} best CV PR-AUC = {s.best_score_:.3f}")

done: A_screening              LogReg        best CV PR-AUC = 0.394


done: A_screening              SVM_RBF       best CV PR-AUC = 0.400


done: A_screening              RandomForest  best CV PR-AUC = 0.396


done: A_screening              HistGB        best CV PR-AUC = 0.386


done: B_plus_symptoms          LogReg        best CV PR-AUC = 0.600


...

done: B_plus_symptoms          HistGB        best CV PR-AUC = 0.573


done: C_plus_cognitive_tests   LogReg        best CV PR-AUC = 0.847


done: C_plus_cognitive_tests   SVM_RBF       best CV PR-AUC = 0.847


done: C_plus_cognitive_tests   RandomForest  best CV PR-AUC = 0.936


done: C_plus_cognitive_tests   HistGB        best CV PR-AUC = 0.948


In [14]:
def clean(v):
    return v.item() if hasattr(v, "item") else v

rows, params_out = [], {}
for (set_name, m), s in tuned.items():
    i = s.best_index_
    d = exp9[(exp9.feature_set == set_name) & (exp9.model == m)].iloc[0]
    best_params = {k.replace("clf__", ""): clean(v) for k, v in s.best_params_.items()}
    params_out[f"{set_name}|{m}"] = best_params
    rows.append({"feature_set": set_name, "model": m,
                 "default CV PR-AUC": round(d.pr_auc, 3),
                 "tuned CV PR-AUC": round(s.cv_results_["mean_test_pr_auc"][i], 3),
                 "tuned SD": round(s.cv_results_["std_test_pr_auc"][i], 3),
                 "gain": round(s.cv_results_["mean_test_pr_auc"][i] - d.pr_auc, 3),
                 "tuned recall": round(s.cv_results_["mean_test_recall"][i], 3),
                 "tuned F1": round(s.cv_results_["mean_test_f1"][i], 3),
                 "best params": best_params})
tune_tbl = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 120)
display(tune_tbl)
tune_tbl.drop(columns="best params").to_csv(OUT_DIR / "stage10_tuning_summary.csv", index=False)
with open(OUT_DIR / "stage10_best_params.json", "w") as f:
    json.dump(params_out, f, indent=2, default=str)

,feature_set,model,default CV PR-AUC,tuned CV PR-AUC,tuned SD,gain,tuned recall,tuned F1,best params
0,A_screening,LogReg,0.394,0.394,0.018,0.000,0.536,0.457,{'C': 0.9846738873614566}
1,A_screening,SVM_RBF,0.386,0.400,0.024,0.014,0.200,0.104,"{'C': 0.017073967431528128, 'gamma': 0.29154431891537513}"
2,A_screening,RandomForest,0.394,0.396,0.014,0.002,0.146,0.218,"{'n_estimators': 200, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 12}"
3,A_screening,HistGB,0.362,0.386,0.022,0.024,0.299,0.336,"{'l2_regularization': 0.03148911647956861, 'learning_rate': 0.2536999076681772, 'max_iter': 300, 'max_leaf_nodes': 8..."
4,B_plus_symptoms,LogReg,0.600,0.600,0.045,0.000,0.579,0.585,{'C': 0.9846738873614566}
5,B_plus_symptoms,SVM_RBF,0.569,0.592,0.039,0.023,0.561,0.585,"{'C': 21.368329072358772, 'gamma': 0.0007068974950624604}"
6,B_plus_symptoms,RandomForest,0.566,0.590,0.033,0.024,0.558,0.582,"{'n_estimators': 200, 'min_samples_leaf': 20, 'max_features': 'sqrt', 'max_depth': 8}"
7,B_plus_symptoms,HistGB,0.546,0.573,0.039,0.027,0.584,0.584,"{'l2_regularization': 0.050190728339613175, 'learning_rate': 0.02028406763400795, 'max_iter': 200, 'max_leaf_nodes':..."
8,C_plus_cognitive_tests,LogReg,0.846,0.847,0.023,0.001,0.827,0.770,{'C': 0.0745934328572655}
9,C_plus_cognitive_tests,SVM_RBF,0.842,0.847,0.026,0.005,0.822,0.763,"{'C': 21.368329072358772, 'gamma': 0.0007068974950624604}"


### Stage 10 result (tuned vs untuned, still CV only)

| Set | Model | Untuned CV PR-AUC | Tuned CV PR-AUC | Gain | Honest reading |
|---|---|---|---|---|---|
| A | LogReg | 0.394 | 0.394 | 0.000 | No change (`C` ≈ 0.98, almost the default of 1) |
| A | SVM | 0.386 | 0.400 | +0.014 | Small lift, inside the SD (0.024); tuned recall collapsed to 0.20 |
| A | RF | 0.394 | 0.396 | +0.002 | Noise |
| A | HistGB | 0.362 | 0.386 | +0.024 | Closes the gap to chance, still not better than LogReg |
| B | LogReg | 0.600 | 0.600 | 0.000 | No change |
| B | SVM / RF / HistGB | 0.569 / 0.566 / 0.546 | 0.592 / 0.590 / 0.573 | +0.02 to +0.03 | Modest; still at or below LogReg 0.600 |
| C | LogReg / SVM | 0.846 / 0.842 | 0.847 / 0.847 | ~0 | Regularisation (`C` = 0.075 on LogReg) did not move PR-AUC |
| C | Random Forest | 0.937 | 0.936 | −0.001 | Defaults were already strong; search picked fewer, shallower trees |
| C | HistGB | 0.943 | **0.948** | +0.005 | Best raw CV score, but the gain is smaller than the SD (0.013) |

**Conclusion:** hyperparameter search did not transform this problem. On sets A and B the features are the bottleneck. On set C the untuned trees were already near the ceiling. That is a valid assignment result: tuning was done correctly and it did not invent a screening signal that is not in the inputs.

Best-CV model by set before the 1-SE rule: A **SVM**, B **LogReg**, C **HistGB**. Stage 11 may still pick a simpler model if it is within one standard error of the best.


---
# Stage 11: Final model, threshold, test evaluation, uncertainty and errors

**Cells in this stage (run in this order):**
1. 1-SE model pick + threshold on training out-of-fold probabilities  
2. Train vs CV vs test table  
3. Final vs baseline on the **test** set, plus confusion counts  
4. Bootstrap 95% confidence intervals  
5. ROC / PR / threshold / calibration plots  
6. Permutation importance (explanation only — not used to drop features)  
7. Error analysis on the primary screening model (set A)  
8. Save `.joblib` models and `thresholds.json`

**Nothing is selected from the test set.** Test predictions start in the first Stage 11 code cell and are only used to *report*.

**How the final model is chosen (CV, not test).** Among algorithms whose tuned CV PR-AUC is within one SD of the best, take the simplest in this order: LogReg → SVM-RBF → Random Forest → HistGB.

**How the threshold is chosen.** Out-of-fold training probabilities are scanned to meet `RECALL_TARGET = 0.90` (catch about 9 in 10 labelled cases). Lowering the threshold raises recall and also raises false alarms. This is a **screening** choice, not a diagnostic one.

**Why these metrics:** the labelled class is the minority (~35%) and a miss is more costly than a false alarm in a screen, so we report PR-AUC, recall, precision, F1, balanced accuracy, ROC-AUC and the confusion matrix. Accuracy alone is not used. Calibration curves ask whether a predicted 0.4 can be read as “about 40% risk”.


In [15]:
FINAL_MODEL_OVERRIDE = {}   # e.g. {"A_screening": "LogReg"}  (only with a written justification)

def pick_final(set_name):
    rows = []
    for m in MODEL_NAMES:
        s = tuned[(set_name, m)]; i = s.best_index_
        rows.append({"model": m, "cv_pr_auc": s.cv_results_["mean_test_pr_auc"][i], "cv_sd": s.cv_results_["std_test_pr_auc"][i]})
    t = pd.DataFrame(rows)
    best = t.loc[t.cv_pr_auc.idxmax()]
    eligible = t[t.cv_pr_auc >= best.cv_pr_auc - best.cv_sd]
    choice = sorted(eligible.model, key=SIMPLICITY.index)[0]
    return FINAL_MODEL_OVERRIDE.get(set_name, choice), t, best.model

def choose_threshold(y, p, target):
    prec, rec, thr_ = precision_recall_curve(y, p)
    ok = np.where(rec[:-1] >= target)[0]
    return float(thr_[ok.max()]) if len(ok) else float(thr_[0])

def metrics_at(y, p, thr):
    y = np.asarray(y); p = np.asarray(p)
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"pr_auc": average_precision_score(y, p), "roc_auc": roc_auc_score(y, p), "brier": brier_score_loss(y, p),
            "threshold": thr, "recall": tp / (tp + fn) if tp + fn else np.nan,
            "precision": tp / (tp + fp) if tp + fp else np.nan, "specificity": tn / (tn + fp) if tn + fp else np.nan,
            "f1": f1_score(y, pred, zero_division=0), "balanced_acc": balanced_accuracy_score(y, pred),
            "accuracy": accuracy_score(y, pred), "TP": tp, "FP": fp, "TN": tn, "FN": fn}

chosen, final_models, thr, oof, test_p, base_models, base_p = {}, {}, {}, {}, {}, {}, {}
sel_rows = []
for set_name in TUNE_SETS:
    name, t, best_name = pick_final(set_name)
    chosen[set_name] = name
    feats = FINAL_SETS[set_name]
    est = tuned[(set_name, name)].best_estimator_
    final_models[set_name] = est
    oof[set_name] = cross_val_predict(clone(est), X_train_F[feats], y_train, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]
    thr[set_name] = choose_threshold(y_train, oof[set_name], RECALL_TARGET)
    bf = FEATURE_SETS[set_name]
    base_models[set_name] = baseline_pipe(bf, "LogReg_default").fit(X_train_F[bf], y_train)
    # ---- test set touched here for the first time: predictions only, nothing is selected from them ----
    test_p[set_name] = est.predict_proba(X_test_F[feats])[:, 1]
    base_p[set_name] = base_models[set_name].predict_proba(X_test_F[bf])[:, 1]
    sel_rows.append({"feature_set": set_name, "best CV PR-AUC model": best_name, "chosen (1-SE rule)": name,
                     "threshold (OOF, recall>=%.2f)" % RECALL_TARGET: round(thr[set_name], 3)})
display(pd.DataFrame(sel_rows))

,feature_set,best CV PR-AUC model,chosen (1-SE rule),"threshold (OOF, recall>=0.90)"
0,A_screening,SVM_RBF,LogReg,0.391
1,B_plus_symptoms,LogReg,LogReg,0.288
2,C_plus_cognitive_tests,HistGB,RandomForest,0.515


In [16]:
# ---------- 11.1 Training vs cross-validation vs test (final model) ----------
cols = ["pr_auc", "roc_auc", "recall", "precision", "f1", "balanced_acc", "specificity"]
rows = []
for set_name in TUNE_SETS:
    feats = FINAL_SETS[set_name]; t_ = thr[set_name]
    ptrain = final_models[set_name].predict_proba(X_train_F[feats])[:, 1]
    for label, y_, p_ in [("train (in-sample)", y_train, ptrain), ("CV out-of-fold", y_train, oof[set_name]),
                          ("TEST (held out)", y_test, test_p[set_name])]:
        m = metrics_at(y_, p_, t_)
        rows.append({"feature_set": set_name, "model": chosen[set_name], "data": label, **{c: round(m[c], 3) for c in cols}})
perf = pd.DataFrame(rows)
display(perf)
perf.to_csv(OUT_DIR / "stage11_train_cv_test.csv", index=False)
print("A large drop from train to CV/test indicates overfitting. Only the TEST rows count as evidence of performance on unseen data.")

,feature_set,model,data,pr_auc,roc_auc,recall,precision,f1,balanced_acc,specificity
0,A_screening,LogReg,train (in-sample),0.430,0.591,0.933,0.365,0.525,0.522,0.112
1,A_screening,LogReg,CV out-of-fold,0.389,0.542,0.901,0.354,0.509,0.501,0.101
2,A_screening,LogReg,TEST (held out),0.404,0.528,0.914,0.349,0.505,0.491,0.068
3,B_plus_symptoms,LogReg,train (in-sample),0.618,0.736,0.938,0.391,0.552,0.570,0.202
4,B_plus_symptoms,LogReg,CV out-of-fold,0.587,0.705,0.901,0.377,0.532,0.544,0.186
5,B_plus_symptoms,LogReg,TEST (held out),0.558,0.678,0.908,0.371,0.527,0.533,0.158
6,C_plus_cognitive_tests,RandomForest,train (in-sample),0.985,0.992,0.926,0.954,0.940,0.951,0.976
7,C_plus_cognitive_tests,RandomForest,CV out-of-fold,0.930,0.956,0.901,0.950,0.925,0.938,0.974
8,C_plus_cognitive_tests,RandomForest,TEST (held out),0.915,0.939,0.921,0.946,0.933,0.946,0.971


A large drop from train to CV/test indicates overfitting. Only the TEST rows count as evidence of performance on unseen data.


In [17]:
# ---------- 11.2 Final vs baseline on the test set ----------
rows = []
for set_name in TUNE_SETS:
    m_final = metrics_at(y_test, test_p[set_name], thr[set_name])
    m_base05 = metrics_at(y_test, base_p[set_name], 0.5)
    rows.append({"feature_set": set_name, "model": f"baseline LogReg (thr 0.5)", **{c: round(m_base05[c], 3) for c in cols}})
    rows.append({"feature_set": set_name, "model": f"final {chosen[set_name]} (thr {thr[set_name]:.2f})", **{c: round(m_final[c], 3) for c in cols}})
cmp11 = pd.DataFrame(rows)
display(cmp11)
cmp11.to_csv(OUT_DIR / "stage11_final_vs_baseline_test.csv", index=False)

# Confusion counts at the chosen threshold
cm_rows = []
for set_name in TUNE_SETS:
    m = metrics_at(y_test, test_p[set_name], thr[set_name])
    cm_rows.append({"feature_set": set_name, "TP": m["TP"], "FN": m["FN"], "FP": m["FP"], "TN": m["TN"]})
display(pd.DataFrame(cm_rows))

,feature_set,model,pr_auc,roc_auc,recall,precision,f1,balanced_acc,specificity
0,A_screening,baseline LogReg (thr 0.5),0.403,0.527,0.033,0.500,0.062,0.507,0.982
1,A_screening,final LogReg (thr 0.39),0.404,0.528,0.914,0.349,0.505,0.491,0.068
2,B_plus_symptoms,baseline LogReg (thr 0.5),0.559,0.677,0.487,0.552,0.517,0.636,0.784
3,B_plus_symptoms,final LogReg (thr 0.29),0.558,0.678,0.908,0.371,0.527,0.533,0.158
4,C_plus_cognitive_tests,baseline LogReg (thr 0.5),0.827,0.887,0.743,0.753,0.748,0.805,0.867
5,C_plus_cognitive_tests,final RandomForest (thr 0.52),0.915,0.939,0.921,0.946,0.933,0.946,0.971


,feature_set,TP,FN,FP,TN
0,A_screening,139,13,259,19
1,B_plus_symptoms,138,14,234,44
2,C_plus_cognitive_tests,140,12,8,270


In [18]:
# ---------- 11.3 Bootstrap 95% confidence intervals on the test set ----------
KEYS = ["pr_auc", "roc_auc", "recall", "precision", "f1", "balanced_acc"]

def bootstrap(y, p_final, p_base, t_final, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    y = np.asarray(y); pf = np.asarray(p_final); pb = np.asarray(p_base)
    vals, diffs = [], []
    for _ in range(n):
        b = rng.integers(0, len(y), len(y))
        if y[b].min() == y[b].max():
            continue
        m = metrics_at(y[b], pf[b], t_final)
        vals.append([m[k] for k in KEYS])
        diffs.append(average_precision_score(y[b], pf[b]) - average_precision_score(y[b], pb[b]))
    return np.array(vals), np.array(diffs)

ci_rows, diff_rows = [], []
for set_name in TUNE_SETS:
    vals, diffs = bootstrap(y_test, test_p[set_name], base_p[set_name], thr[set_name])
    pt = metrics_at(y_test, test_p[set_name], thr[set_name])
    lo, hi = np.nanpercentile(vals, [2.5, 97.5], axis=0)
    row = {"feature_set": set_name, "model": chosen[set_name]}
    for k, l, h in zip(KEYS, lo, hi):
        row[k] = f"{pt[k]:.3f} [{l:.3f}, {h:.3f}]"
    ci_rows.append(row)
    dl, dh = np.percentile(diffs, [2.5, 97.5])
    diff_rows.append({"feature_set": set_name, "PR-AUC gain vs baseline (point)":
                      round(average_precision_score(y_test, test_p[set_name]) - average_precision_score(y_test, base_p[set_name]), 3),
                      "95% CI": f"[{dl:.3f}, {dh:.3f}]", "share of resamples > 0": round((diffs > 0).mean(), 3)})
display(pd.DataFrame(ci_rows))
display(pd.DataFrame(diff_rows))
pd.DataFrame(ci_rows).to_csv(OUT_DIR / "stage11_test_bootstrap_ci.csv", index=False)
print("If the CI of the gain over the baseline includes 0, do not claim the final model is better.")

,feature_set,model,pr_auc,roc_auc,recall,precision,f1,balanced_acc
0,A_screening,LogReg,"0.404 [0.336, 0.486]","0.528 [0.470, 0.592]","0.914 [0.867, 0.956]","0.349 [0.302, 0.397]","0.505 [0.452, 0.557]","0.491 [0.464, 0.518]"
1,B_plus_symptoms,LogReg,"0.558 [0.480, 0.646]","0.678 [0.620, 0.732]","0.908 [0.858, 0.950]","0.371 [0.324, 0.419]","0.527 [0.474, 0.577]","0.533 [0.501, 0.564]"
2,C_plus_cognitive_tests,RandomForest,"0.915 [0.859, 0.960]","0.939 [0.907, 0.967]","0.921 [0.875, 0.963]","0.946 [0.902, 0.979]","0.933 [0.901, 0.962]","0.946 [0.922, 0.970]"


,feature_set,PR-AUC gain vs baseline (point),95% CI,share of resamples > 0
0,A_screening,0.001,"[-0.004, 0.006]",0.648
1,B_plus_symptoms,-0.001,"[-0.004, 0.002]",0.310
2,C_plus_cognitive_tests,0.088,"[0.038, 0.143]",0.999


If the CI of the gain over the baseline includes 0, do not claim the final model is better.


In [19]:
# ---------- 11.4 Curves, threshold, confusion matrices, calibration ----------
n = len(TUNE_SETS)  # used by the plots in this cell and the next
fig, axes = plt.subplots(2, n, figsize=(5 * n, 8.5), squeeze=False)
for j, s in enumerate(TUNE_SETS):
    for p_, lab, ls in [(test_p[s], f"final {chosen[s]}", "-"), (base_p[s], "baseline LogReg", "--")]:
        fpr, tpr, _ = roc_curve(y_test, p_)
        axes[0, j].plot(fpr, tpr, ls, label=f"{lab} (AUC {roc_auc_score(y_test, p_):.2f})")
        pr, rc, _ = precision_recall_curve(y_test, p_)
        axes[1, j].plot(rc, pr, ls, label=f"{lab} (AP {average_precision_score(y_test, p_):.2f})")
    axes[0, j].plot([0, 1], [0, 1], color="grey", lw=0.8); axes[0, j].set_title(f"ROC: {s}"); axes[0, j].set_xlabel("FPR"); axes[0, j].set_ylabel("TPR")
    axes[1, j].axhline(y_test.mean(), color="grey", lw=0.8); axes[1, j].set_title(f"Precision-Recall: {s}"); axes[1, j].set_xlabel("Recall"); axes[1, j].set_ylabel("Precision")
    axes[0, j].legend(fontsize=8); axes[1, j].legend(fontsize=8)
savefig("11_roc_pr_test")

fig, axes = plt.subplots(1, n, figsize=(4.6 * n, 4), squeeze=False)
for j, s in enumerate(TUNE_SETS):
    m = metrics_at(y_test, test_p[s], thr[s])
    sns.heatmap([[m["TN"], m["FP"]], [m["FN"], m["TP"]]], annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0, j],
                xticklabels=["Pred no", "Pred yes"], yticklabels=["True no", "True yes"])
    axes[0, j].set_title(f"{s}\n{chosen[s]}, threshold {thr[s]:.2f}")
savefig("11_confusion_matrices")

fig, axes = plt.subplots(1, n, figsize=(4.6 * n, 4), squeeze=False)
for j, s in enumerate(TUNE_SETS):
    prec, rec, th = precision_recall_curve(y_train, oof[s])
    axes[0, j].plot(th, rec[:-1], label="recall"); axes[0, j].plot(th, prec[:-1], label="precision")
    axes[0, j].axvline(thr[s], color="red", ls="--", lw=1, label=f"chosen {thr[s]:.2f}")
    axes[0, j].axhline(RECALL_TARGET, color="grey", ls=":", lw=1)
    axes[0, j].set_title(f"Threshold choice (train OOF): {s}"); axes[0, j].set_xlabel("threshold"); axes[0, j].legend(fontsize=8)
savefig("11_threshold_choice")

fig, axes = plt.subplots(1, n, figsize=(4.6 * n, 4), squeeze=False)
for j, s in enumerate(TUNE_SETS):
    fr, mp = calibration_curve(y_test, test_p[s], n_bins=8, strategy="quantile")
    axes[0, j].plot(mp, fr, marker="o", label=f"Brier {brier_score_loss(y_test, test_p[s]):.3f}"); axes[0, j].plot([0, 1], [0, 1], color="grey", lw=0.8)
    axes[0, j].set_title(f"Calibration (test): {s}"); axes[0, j].set_xlabel("mean predicted probability"); axes[0, j].set_ylabel("observed fraction"); axes[0, j].legend(fontsize=8)
savefig("11_calibration")

In [20]:
# ---------- 11.5 Explainability: permutation importance on the test set (explanation only, no selection) ----------
perm = {}
for s in TUNE_SETS:
    feats = FINAL_SETS[s]
    r = permutation_importance(final_models[s], X_test_F[feats], y_test, scoring="average_precision",
                               n_repeats=5 if FAST_MODE else 20, random_state=SEED, n_jobs=-1)
    perm[s] = pd.DataFrame({"feature": feats, "importance": r.importances_mean, "sd": r.importances_std}).sort_values("importance", ascending=False)

fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 5), squeeze=False)
for j, s in enumerate(TUNE_SETS):
    d = perm[s].head(12).iloc[::-1]
    axes[0, j].barh(d.feature, d.importance, xerr=d.sd, color="#5b8fb9")
    axes[0, j].set_title(f"{s}\n({chosen[s]})", fontsize=10); axes[0, j].set_xlabel("drop in PR-AUC when shuffled")
savefig("11_permutation_importance")
display(perm[PRIMARY_SET].head(10).round(4))

# Logistic regression: readable coefficients (standardised features -> comparable magnitudes)
for s in TUNE_SETS:
    if chosen[s] == "LogReg":
        est = final_models[s]
        names = est.named_steps["prep"].get_feature_names_out()
        if "select" in est.named_steps:
            names = names[est.named_steps["select"].get_support()]
        coef = pd.Series(est.named_steps["clf"].coef_[0], index=names).sort_values(key=abs, ascending=False)
        print(f"LogReg coefficients, {s} (log-odds per 1 SD / per unit for 0-1 features), top 10:")
        display(coef.head(10).round(3).to_frame("coef"))

,feature,importance,sd
9,SleepQuality,0.0198,0.0113
2,Ethnicity,0.0144,0.0200
15,Hypertension,0.0130,0.0075
3,EducationLevel,0.0130,0.0095
14,HeadInjury,0.0082,0.0061
21,CholesterolTriglycerides,0.0065,0.0075
20,CholesterolHDL,0.0056,0.0123
16,SystolicBP,0.0056,0.0112
13,Depression,0.0029,0.0012
10,FamilyHistoryAlzheimers,0.0017,0.0066


LogReg coefficients, A_screening (log-odds per 1 SD / per unit for 0-1 features), top 10:


,coef
Ethnicity_2,0.316
CardiovascularDisease,0.254
HeadInjury,-0.238
Diabetes,-0.210
Hypertension,0.206
Ethnicity_3,-0.186
FamilyHistoryAlzheimers,-0.147
Ethnicity_1,-0.123
SleepQuality,-0.119
CholesterolHDL,0.112


LogReg coefficients, B_plus_symptoms (log-odds per 1 SD / per unit for 0-1 features), top 10:


,coef
MemoryComplaints,1.710
BehavioralProblems,1.338
HeadInjury,-0.315
Hypertension,0.292
Ethnicity_3,-0.281
CardiovascularDisease,0.270
PersonalityChanges,-0.207
Ethnicity_1,-0.206
Diabetes,-0.167
Ethnicity_2,0.165


In [21]:
# ---------- 11.6 Error analysis (primary screening model) ----------
pm = PRIMARY_SET
te = X_test_F.copy()
te["y"] = y_test.values; te["p"] = test_p[pm]; te["pred"] = (te["p"] >= thr[pm]).astype(int)
te["outcome"] = np.select([(te.y == 1) & (te.pred == 1), (te.y == 1) & (te.pred == 0), (te.y == 0) & (te.pred == 1)],
                          ["TP", "FN", "FP"], "TN")
print(f"Error analysis for {pm} ({chosen[pm]}), threshold {thr[pm]:.3f}")
display(te.outcome.value_counts().to_frame("count").T)

top_feats = perm[pm].feature.head(6).tolist()
display(te.groupby("outcome")[top_feats].mean().T.round(2))

print("Most confidently MISSED cases (false negatives, lowest predicted risk):")
display(te[te.outcome == "FN"].sort_values("p").head(8)[["p"] + top_feats].round(3))
print("Most confident FALSE ALARMS (false positives, highest predicted risk):")
display(te[te.outcome == "FP"].sort_values("p", ascending=False).head(8)[["p"] + top_feats].round(3))

Error analysis for A_screening (LogReg), threshold 0.391


outcome,FP,TP,TN,FN
count,259,139,19,13


outcome,FN,FP,TN,TP
SleepQuality,8.26,7.11,7.80,6.80
Ethnicity,1.08,0.65,1.21,0.61
Hypertension,0.15,0.15,0.05,0.19
EducationLevel,1.54,1.29,1.89,1.17
HeadInjury,0.23,0.09,0.16,0.09
CholesterolTriglycerides,128.00,232.35,191.32,227.26


Most confidently MISSED cases (false negatives, lowest predicted risk):


,p,SleepQuality,Ethnicity,Hypertension,EducationLevel,HeadInjury,CholesterolTriglycerides
327,0.262,6.203,1,0,3,0,150.955
315,0.301,8.100,1,0,2,0,90.879
214,0.335,9.689,0,0,2,1,108.758
65,0.338,7.067,1,0,1,1,68.967
204,0.350,6.992,1,0,1,0,78.382
251,0.359,8.404,3,0,1,0,368.022
324,0.362,9.536,3,0,0,0,54.564
134,0.367,9.989,1,0,3,0,76.566


Most confident FALSE ALARMS (false positives, highest predicted risk):


,p,SleepQuality,Ethnicity,Hypertension,EducationLevel,HeadInjury,CholesterolTriglycerides
48,0.672,7.687,0,1,1,0,379.354
28,0.657,6.226,2,0,2,0,90.425
283,0.656,4.612,0,0,0,0,171.707
253,0.651,7.837,2,1,1,0,313.053
74,0.649,4.141,0,0,1,0,102.880
154,0.643,8.129,2,0,0,1,215.982
213,0.643,6.854,0,0,1,0,290.913
339,0.637,6.136,0,0,1,0,191.113


In [22]:
# ---------- 11.7 Save final models and metrics ----------
for s in TUNE_SETS:
    joblib.dump({"model": final_models[s], "features": FINAL_SETS[s], "threshold": thr[s], "mode": s,
                 "algorithm": chosen[s], "recall_target": RECALL_TARGET, "sklearn_version": sklearn.__version__, "seed": SEED},
                MODEL_DIR / f"final_{s}.joblib")
with open(OUT_DIR / "thresholds.json", "w") as f:
    json.dump({s: {"algorithm": chosen[s], "threshold": float(thr[s])} for s in TUNE_SETS}, f, indent=2)
print("Saved:", sorted(p.name for p in MODEL_DIR.glob("*.joblib")))

Saved: ['final_A_screening.joblib', 'final_B_plus_symptoms.joblib', 'final_C_plus_cognitive_tests.joblib']


### Stage 11 result (final evaluation)

**1. Which model was chosen, and how?**  
From tuned **CV** and the 1-SE rule (not the test set):

| Feature set | Best CV PR-AUC | Chosen (1-SE, simpler) | Threshold (train OOF, recall ≥ 0.90) |
|---|---|---|---|
| A screening | SVM | **LogReg** | 0.39 |
| B + symptoms | LogReg | **LogReg** | 0.29 |
| C + cognitive tests | HistGB | **Random Forest** | 0.52 |

On set A, SVM’s tuned PR-AUC was only fractionally higher and its recall was poor, so LogReg is the defensible screen. On set C, HistGB was best (0.948) but Random Forest (0.936) sits inside one SD, so the simpler tree ensemble is kept.

**2. Is it better than the baseline on unseen data?**  
Test PR-AUC vs default LogReg at threshold 0.5:

| Set | Baseline PR-AUC | Final PR-AUC | Bootstrap 95% CI of the **gain** | Claim allowed? |
|---|---|---|---|---|
| A | 0.403 | 0.404 | **[−0.004, 0.006]** (64.8% of resamples > 0) | **No** — CI includes 0 |
| B | 0.559 | 0.558 | **[−0.004, 0.002]** | **No** |
| C | 0.827 | **0.915** | **[+0.038, +0.143]** (99.9% > 0) | **Yes** — Random Forest is better |

The screening models are **not demonstrably better** than a default logistic regression. They mainly change the *operating point* (much higher recall, many more false alarms). Set C is a different task: once cognitive tests are in the table, the forest clearly outranks the linear baseline.

**3. Train vs CV vs test (overfitting).**  
Set A LogReg: train PR-AUC 0.43 → CV 0.39 → test 0.40 (stable, weak).  
Set B LogReg: 0.62 → 0.59 → 0.56 (small drop; still usable ranking, not a screen you would trust alone).  
Set C Random Forest: train 0.985 → CV 0.93 → test **0.915** (some optimism on train, but test remains high).

**4. Threshold trade-off on the test set (430 patients, 152 labelled positive).**

| Set | TP | FN | FP | TN | Recall | Precision | Specificity |
|---|---|---|---|---|---|---|---|
| A LogReg, thr 0.39 | 139 | 13 | 259 | 19 | 0.914 | 0.349 | 0.068 |
| B LogReg, thr 0.29 | 138 | 14 | 234 | 44 | 0.908 | 0.371 | 0.158 |
| C RF, thr 0.52 | 140 | 12 | 8 | 270 | 0.921 | 0.946 | 0.971 |

The 0.90 recall target **is met**, but on sets A and B almost every person is flagged (specificity 7% and 16%). That is the cost of forcing high recall when ranking quality is near chance. Set C meets the recall target with almost no extra false alarms.

**5. Calibration.** Set A probabilities are barely informative (test ROC-AUC 0.53), so they should not be read as personalised risk. Set C is much better ranked (ROC-AUC 0.94); even then the number is a **score**, not a diagnosis.

**6. Errors on the primary screen (set A).** 13 missed cases (FN) tend to have better sleep scores and fewer of the weak flags the model uses. 259 false alarms are the default behaviour of a low threshold on a weak model — not a specific clinical subtype. Coefficients also disagree with textbook risk (for example family history and diabetes get negative weights). Treat them as **dataset artefacts**, not biology.

**7. Explainability.** Permutation importance on set A is small and noisy (sleep, ethnicity, hypertension). On set B, `MemoryComplaints` and `BehavioralProblems` dominate the LogReg coefficients (about +1.7 and +1.3 log-odds). On set C the forest is driven by MMSE / functional / ADL — which matches the mutual-information ranking and also shows why set C is “too late” to call early screening.

**8. Unseen patients.** Performance can only be claimed on this synthetic table. Width of the CIs (set A PR-AUC 0.40 [0.34, 0.49]) already shows the test set is small. Stage 12 checks missing inputs and subgroups without changing the model.

**9. A vs B vs C.** Cheap screening inputs do not rank diagnosis. Reported symptoms help ranking (PR-AUC ~0.56) but not a clean screen at 90% recall. Cognitive tests make the problem easy — and they are the inputs a specialist would already collect. DementiaSense is therefore a **feature-availability** result more than an algorithm result.


---
# Stage 12: Use case, impact and limitations

**Cells in this stage:** missing-data robustness on the **already chosen** test predictions → subgroup table (descriptive only) → inference demo from the saved `.joblib` → written use case.

These checks **do not** change the Stage 11 model. They show how a frozen tool behaves when inputs are incomplete or when the sample is sliced. They are not a second round of model selection.

**Primary demo mode:** `PRIMARY_SET = "A_screening"` — the cheap-to-collect view. Sets B and C are comparison modes, not the community-screening claim.


In [23]:
# ---------- 12.1 Robustness to incomplete inputs (simulated missing values in the TEST data) ----------
rng = np.random.default_rng(SEED)
rows = []
for s in TUNE_SETS:
    feats = FINAL_SETS[s]
    for frac in [0.0, 0.1, 0.2, 0.3]:
        Xm = X_test[ALL_FEATURES].copy()
        if frac > 0:
            Xm = Xm.mask(rng.random(Xm.shape) < frac)
        p_ = final_models[s].predict_proba(add_features(Xm)[feats])[:, 1]
        m = metrics_at(y_test, p_, thr[s])
        rows.append({"feature_set": s, "missing %": int(frac * 100), "pr_auc": round(m["pr_auc"], 3),
                     "roc_auc": round(m["roc_auc"], 3), "recall": round(m["recall"], 3), "precision": round(m["precision"], 3)})
rob = pd.DataFrame(rows)
display(rob)
rob.to_csv(OUT_DIR / "stage12_missing_data_robustness.csv", index=False)
print("Values are blanked at random and imputed by the pipeline (median / most frequent from the TRAINING data).")

,feature_set,missing %,pr_auc,roc_auc,recall,precision
0,A_screening,0,0.404,0.528,0.914,0.349
1,A_screening,10,0.401,0.536,0.921,0.351
2,A_screening,20,0.383,0.509,0.954,0.351
3,A_screening,30,0.398,0.526,0.941,0.346
4,B_plus_symptoms,0,0.558,0.678,0.908,0.371
5,B_plus_symptoms,10,0.551,0.672,0.901,0.360
6,B_plus_symptoms,20,0.512,0.627,0.875,0.345
7,B_plus_symptoms,30,0.515,0.633,0.901,0.353
8,C_plus_cognitive_tests,0,0.915,0.939,0.921,0.946
9,C_plus_cognitive_tests,10,0.865,0.916,0.842,0.859


Values are blanked at random and imputed by the pipeline (median / most frequent from the TRAINING data).


In [24]:
# ---------- 12.2 Subgroup performance (primary screening model) ----------
sg = X_test_F[["Gender", "Ethnicity", "EducationLevel", "Age"]].copy()
sg["AgeGroup"] = pd.cut(sg["Age"], bins=[0, 69, 79, 200], labels=["<70", "70-79", "80+"])
sg["y"] = y_test.values; sg["p"] = test_p[PRIMARY_SET]

rows = []
for col in ["Gender", "Ethnicity", "EducationLevel", "AgeGroup"]:
    for level, g in sg.groupby(col, observed=True):
        n_pos = int(g.y.sum())
        enough = len(g) >= 30 and 5 <= n_pos <= len(g) - 5
        m = metrics_at(g.y, g.p, thr[PRIMARY_SET]) if enough else None
        rows.append({"variable": col, "level": level, "n": len(g), "positives": n_pos,
                     "recall": round(m["recall"], 3) if m else np.nan,
                     "precision": round(m["precision"], 3) if m else np.nan,
                     "pr_auc": round(m["pr_auc"], 3) if m else np.nan})
subg = pd.DataFrame(rows)
display(subg)
subg.to_csv(OUT_DIR / "stage12_subgroup_performance.csv", index=False)
print("Subgroups are small, so differences are noisy. Use this table to flag where to look, not to prove fairness or bias.")

,variable,level,n,positives,recall,precision,pr_auc
0,Gender,0,210,80,0.925,0.374,0.425
1,Gender,1,220,72,0.903,0.325,0.400
2,Ethnicity,0,263,97,0.969,0.373,0.406
3,Ethnicity,1,80,24,0.667,0.239,0.382
4,Ethnicity,2,50,18,1.000,0.360,0.530
5,Ethnicity,3,37,13,0.846,0.379,0.432
6,EducationLevel,0,90,33,0.939,0.360,0.464
7,EducationLevel,1,174,67,0.925,0.378,0.435
8,EducationLevel,2,118,40,0.925,0.336,0.381
9,EducationLevel,3,48,12,0.750,0.237,0.271


Subgroups are small, so differences are noisy. Use this table to flag where to look, not to prove fairness or bias.


In [25]:
# ---------- 12.3 Inference demo: how the saved model would score a NEW patient ----------
def predict_risk(patient, mode=PRIMARY_SET):
    path = MODEL_DIR / f"final_{mode}.joblib"
    if not path.exists():
        raise FileNotFoundError(f"Saved model not found: {path}. Run Stage 11 (save cell) first.")
    art = joblib.load(path)
    row = pd.DataFrame([patient]).reindex(columns=ALL_FEATURES).apply(pd.to_numeric, errors="coerce")
    row = add_features(row)                               # missing inputs become NaN and are imputed by the pipeline
    missing_for_model = [c for c in art["features"] if c not in row.columns]
    if missing_for_model:
        raise KeyError(f"Engineered/original features missing after add_features: {missing_for_model}")
    p = float(art["model"].predict_proba(row[art["features"]])[:, 1][0])
    flagged = p >= art["threshold"]
    return {"mode": mode, "model": art["algorithm"], "risk_probability": round(p, 3),
            "threshold": round(art["threshold"], 3),
            "recommendation": ("REFER: higher estimated risk; suggest specialist cognitive assessment" if flagged
                               else "Lower estimated risk: routine monitoring"),
            "note": "Screening estimate only. Not a medical diagnosis. A clinician makes the decision."}

demo = X_test[ALL_FEATURES].iloc[:3]
for i in range(3):
    print("Actual label (for reference):", int(y_test.iloc[i]), "->", predict_risk(demo.iloc[i].to_dict()))
print()
print("Same patient with several inputs missing:")
partial = demo.iloc[0].to_dict()
for k in list(partial)[:12]:
    partial.pop(k)
print(predict_risk(partial))

Actual label (for reference): 1 -> {'mode': 'A_screening', 'model': 'LogReg', 'risk_probability': 0.585, 'threshold': 0.391, 'recommendation': 'REFER: higher estimated risk; suggest specialist cognitive assessment', 'note': 'Screening estimate only. Not a medical diagnosis. A clinician makes the decision.'}
Actual label (for reference): 1 -> {'mode': 'A_screening', 'model': 'LogReg', 'risk_probability': 0.441, 'threshold': 0.391, 'recommendation': 'REFER: higher estimated risk; suggest specialist cognitive assessment', 'note': 'Screening estimate only. Not a medical diagnosis. A clinician makes the decision.'}


Actual label (for reference): 1 -> {'mode': 'A_screening', 'model': 'LogReg', 'risk_probability': 0.583, 'threshold': 0.391, 'recommendation': 'REFER: higher estimated risk; suggest specialist cognitive assessment', 'note': 'Screening estimate only. Not a medical diagnosis. A clinician makes the decision.'}

Same patient with several inputs missing:
{'mode': 'A_screening', 'model': 'LogReg', 'risk_probability': 0.536, 'threshold': 0.391, 'recommendation': 'REFER: higher estimated risk; suggest specialist cognitive assessment', 'note': 'Screening estimate only. Not a medical diagnosis. A clinician makes the decision.'}


### 12.4 Use case, impact and limitations

**Who it is for.** A GP or community worker who can collect age, demographics, lifestyle, history and routine vitals/labs (set A), optionally plus reported symptoms (set B). Memory clinics that already record MMSE, functional assessment and ADL are set C — that is **decision support after tests**, not early triage.

**How it would be used.** A row of those fields goes into `predict_risk()`. The saved pipeline imputes blanks the same way as training (median / most frequent), returns a probability, and compares it with the frozen threshold. Output is **refer vs routine monitoring**, plus the explicit note that a clinician decides. The demo cell scores three held-out rows and the same row with 12 fields removed; the pipeline still returns a probability.

**Practical impact, from the test confusion matrix (per 100 people, using the observed test mix).**

| Mode | True cases caught (TP) | True cases missed (FN) | False alarms (FP) | Correctly cleared (TN) |
|---|---|---|---|---|
| A, thr 0.39 | ~32 | ~3 | ~60 | ~4 |
| B, thr 0.29 | ~32 | ~3 | ~54 | ~10 |
| C, thr 0.52 | ~33 | ~3 | ~2 | ~63 |

Set A would refer most people to catch 32 of ~35 labelled cases. That is not a usable community screen: the false-alarm load would swamp a clinic. Set C looks clinically tidy **only because cognitive tests are already in the input**.

**Missing-data robustness.** Randomly blanking 10–30% of test fields barely moves set A (PR-AUC stays ~0.38–0.40) because there was little signal to lose. Set C falls from **0.915 → 0.800** at 20% missing: the forest depends on those cognitive scores. Incomplete clinic forms would hurt the only strong model.

**Subgroups (set A, descriptive).** Gender is similar. Some ethnicity and education slices look different (for example ethnicity level 1 recall 0.67 on n = 80; education level 3 PR-AUC 0.27 on n = 48). Those groups are too small to prove bias. They are flags for a larger study, not fairness conclusions.

**Risks if someone treated this as a medical device.** False negatives give false reassurance. False positives (the main failure of A/B) cause anxiety and unnecessary referrals. Over-reliance, privacy of health data, and the fact that several “risk factors” in the LogReg have the wrong sign on this table.

**Limitations (must be stated).**
- The Kaggle table is **synthetically generated** for education, not real clinical records.
- One dataset, one 80/20 split, **no external validation**.
- Test n = 430, so confidence intervals are wide.
- Population, geography and how `Diagnosis` was assigned are unknown.
- MMSE / ADL / functional assessment are close to the label — using them is not early detection.
- Correlation is not causation; permutation importance is not a biological mechanism.
- No prospective, regulatory or usability study.

**What we are not claiming.** Deployment, clinical validity, or real-world lives saved. The honest product of this assignment is: *on this table, algorithm choice does not create an early screen; input timing does.*

**Future work (only if a real cohort existed).** External validation (patient-level split), proper calibration, biomarkers or imaging, and a clinician-facing interface with drift monitoring. None of that is demonstrated here.


---
# Reproducibility and academic-integrity record

The next cell prints the exact seed, library versions, frozen decisions (`USE_ENGINEERED`, `SELECT_K`, balancing, chosen models, thresholds) and every file written under `outputs/`. Those numbers are the ones used in the stage write-ups above.


In [26]:
print("Seed:", SEED, "| Python", platform.python_version(), "| scikit-learn", sklearn.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__)
print("imbalanced-learn available:", HAS_IMBLEARN)
print()
print("Decisions taken in this notebook:")
print("  Balancing:", BALANCING, "| engineered features used:", USE_ENGINEERED, "| select_k:", SELECT_K)
print("  Final models:", chosen)
print("  Thresholds (recall target %.2f):" % RECALL_TARGET, {k: round(v, 3) for k, v in thr.items()})
print()
print("Files written to", OUT_DIR.resolve())
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file() and p.suffix in (".csv", ".json", ".joblib", ".png"):
        print(" ", p.relative_to(OUT_DIR))

Seed: 42 | Python 3.10.7 | scikit-learn 1.7.2 | pandas 2.3.3 | numpy 2.2.6
imbalanced-learn available: False

Decisions taken in this notebook:
  Balancing: class_weight | engineered features used: False | select_k: None
  Final models: {'A_screening': 'LogReg', 'B_plus_symptoms': 'LogReg', 'C_plus_cognitive_tests': 'RandomForest'}
  Thresholds (recall target 0.90): {'A_screening': 0.391, 'B_plus_symptoms': 0.288, 'C_plus_cognitive_tests': 0.515}

Files written to /Users/hirenrathva/DementiaSense/outputs
  .mplconfig/fontlist-v390.json
  feature_config.json
  figures/04_outlier_boxplots.png
  figures/05_class_distribution.png
  figures/06_age_trend.png
  figures/06_binary_prevalence.png
  figures/06_correlation_heatmap.png
  figures/06_diagnosis_rate_by_category.png
  figures/06_top_continuous_by_target.png
  figures/06_univariate_continuous.png
  figures/07_mutual_information.png
  figures/09_model_comparison.png
  figures/11_calibration.png
  figures/11_confusion_matrices.png
  figur

**Record of resources**

| Resource | Details |
|---|---|
| Dataset | Alzheimer's Disease Dataset (Rabie El Kharoua, 2024). Kaggle: `https://www.kaggle.com/datasets/rabieelkharoua/alzheimers-disease-dataset`. DOI: `10.34740/KAGGLE/DSV/8668279`. Licence: CC BY 4.0. **Synthetically generated** for education; not real clinical records. File: `alzheimers_disease_data.csv` (2,149 rows). |
| Libraries | pandas 2.3.3, numpy 2.2.6, scikit-learn 1.7.2, scipy, matplotlib, seaborn, joblib. imbalanced-learn is optional and was **not** required (`BALANCING = class_weight`). |
| Split / seed | Stratified 80/20 in Stage 4; `SEED = 42` from `feature_config.json`. Train 1,719 rows (35.4% positive), test 430 rows (35.3% positive). |
| Code / assistance | Notebook structure follows the assignment stages. Narrative in this file is written from the executed outputs in `outputs/`. |
| References | Dataset data card (Kaggle / DOI above). |
